In [ ]:
# Cell 1 — Install & imports
!pip install scikit-learn pandas numpy joblib matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import re
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded")

✅ All libraries loaded


In [ ]:
# Cell 2 — Upload files
from google.colab import files

print("Upload your 4 CSV files now:")
print("1. freelancer_earnings_bd.csv")
print("2. upwork_data_scientists.csv")
print("3. JobsDatasetProcessed.csv")
print("4. upwork-jobs.csv")

uploaded = files.upload()
print("\n✅ Files uploaded:", list(uploaded.keys()))

Upload your 4 CSV files now:
1. freelancer_earnings_bd.csv
2. upwork_data_scientists.csv
3. JobsDatasetProcessed.csv
4. upwork-jobs.csv


Saving freelancer_earnings_bd.csv to freelancer_earnings_bd (1).csv
Saving JobsDatasetProcessed.csv to JobsDatasetProcessed.csv
Saving upwork_data_scientists.csv to upwork_data_scientists.csv
Saving upwork-jobs_final.csv to upwork-jobs_final.csv

✅ Files uploaded: ['freelancer_earnings_bd (1).csv', 'JobsDatasetProcessed.csv', 'upwork_data_scientists.csv', 'upwork-jobs_final.csv']


In [ ]:
# Cell 3 — Load datasets
df_earnings   = pd.read_csv('freelancer_earnings_bd.csv')
df_upwork_ds  = pd.read_csv('upwork_data_scientists.csv')
df_jobs       = pd.read_csv('JobsDatasetProcessed.csv')
df_upwork     = pd.read_csv('upwork-jobs_final.csv')

print("freelancer_earnings_bd  :", df_earnings.shape)
print("upwork_data_scientists  :", df_upwork_ds.shape)
print("JobsDatasetProcessed    :", df_jobs.shape)
print("upwork-jobs             :", df_upwork.shape)

freelancer_earnings_bd  : (1950, 15)
upwork_data_scientists  : (132, 10)
JobsDatasetProcessed    : (3000, 9)
upwork-jobs             : (53058, 9)


In [ ]:
# Cell 4 — Clean df_earnings
df1 = df_earnings.copy()

# Standardise experience level to lowercase
df1['experience_level'] = df1['Experience_Level'].str.lower().str.strip()

# Keep only IT-relevant categories
it_categories = [
    'Web Development', 'App Development', 'Data Entry',
    'Digital Marketing', 'SEO', 'Content Writing'
]
df1 = df1[df1['Job_Category'].isin(it_categories)].reset_index(drop=True)

# Rename columns to snake_case
df1 = df1.rename(columns={
    'Job_Category'    : 'job_category',
    'Job_Completed'   : 'jobs_completed',
    'Earnings_USD'    : 'earnings_usd',
    'Hourly_Rate'     : 'hourly_rate',
    'Job_Success_Rate': 'job_success_rate',
    'Client_Rating'   : 'client_rating',
    'Job_Duration_Days': 'job_duration_days',
    'Rehire_Rate'     : 'rehire_rate',
})

# Create binary label: 1 = beginner, 0 = not beginner
df1['is_beginner'] = (df1['experience_level'] == 'beginner').astype(int)

# Keep only useful columns
df1 = df1[[
    'job_category','experience_level','is_beginner',
    'jobs_completed','earnings_usd','hourly_rate',
    'job_success_rate','client_rating','job_duration_days','rehire_rate'
]]

print("df1 shape:", df1.shape)
print("\nExperience distribution:")
print(df1['experience_level'].value_counts())
print("\nSample:")
print(df1.head(3))

df1 shape: (1441, 10)

Experience distribution:
experience_level
beginner        497
intermediate    482
expert          462
Name: count, dtype: int64

Sample:
      job_category experience_level  is_beginner  jobs_completed  \
0  Web Development         beginner            1             180   
1  App Development         beginner            1             218   
2  Web Development         beginner            1              27   

   earnings_usd  hourly_rate  job_success_rate  client_rating  \
0          1620        95.79             68.73           3.18   
1          9078        86.38             97.54           3.44   
2          3455        85.17             86.60           4.20   

   job_duration_days  rehire_rate  
0                  1        40.19  
1                 54        36.53  
2                 46        74.05  


In [ ]:
# Cell 5 — Clean df_upwork_ds
df2 = df_upwork_ds.copy()

# Clean jobSuccess — remove % sign
df2['job_success_pct'] = (
    df2['jobSuccess']
    .str.replace('%','', regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors='coerce')
)

# totalJobs → numeric
df2['total_jobs'] = pd.to_numeric(df2['totalJobs'], errors='coerce').fillna(0)

# Split pipe-separated skills into a list, then explode for analysis
df2['skills_list'] = df2['skills'].str.split('|')

# Label beginners: totalJobs < 5 OR jobSuccess missing
df2['is_beginner'] = (
    (df2['total_jobs'] < 5) |
    (df2['job_success_pct'].isna())
).astype(int)

df2 = df2[['hourlyRate','job_success_pct','total_jobs',
           'skills_list','is_beginner']].dropna(subset=['hourlyRate'])

print("df2 shape:", df2.shape)
print("\nBeginner distribution:")
print(df2['is_beginner'].value_counts())
print("\nSample:")
print(df2.head(3))

df2 shape: (132, 5)

Beginner distribution:
is_beginner
0    115
1     17
Name: count, dtype: int64

Sample:
   hourlyRate  job_success_pct  total_jobs  \
0        25.0             97.0          69   
1        15.0             97.0          68   
2        60.0            100.0         351   

                                         skills_list  is_beginner  
0  [Python, Machine Learning, Scrapy, Data Scienc...            0  
1  [Data Entry, Google Docs, Online Research, Lea...            0  
2  [Microsoft Excel, Excel Formula, Excel Macros,...            0  


In [ ]:
# Cell 6 — Clean df_jobs
df3 = df_jobs.copy()

# Drop rows with missing IT Skills
df3 = df3.dropna(subset=['IT Skills']).reset_index(drop=True)

# Rename columns
df3 = df3.rename(columns={
    'Query'    : 'job_category',
    'Job Title': 'job_title',
    'IT Skills': 'it_skills',
})

# Keep only useful columns
df3 = df3[['job_category','job_title','it_skills']].copy()

# Build a flat skill list per category (used later for matching)
df3['skills_list'] = df3['it_skills'].str.split(',').apply(
    lambda x: [s.strip().lower() for s in x] if isinstance(x, list) else []
)

print("df3 shape:", df3.shape)
print("\nCategories:")
print(df3['job_category'].value_counts().head(10))

df3 shape: (2990, 4)

Categories:
job_category
Artificial Intelligence          120
Business Intelligence Analyst    120
Data Architect                   120
Data Analyst                     120
Cloud Services Developer         120
Data Warehousing                 120
Data Visualization Expert        120
Data Scientist                   120
Data Quality Manager             120
Technology Integration           120
Name: count, dtype: int64


In [ ]:
# Cell 7 — Clean df_upwork (job postings)
df4 = df_upwork.copy()

# Drop rows with no title or description
df4 = df4.dropna(subset=['title','description']).reset_index(drop=True)

# Extract budget — use budget col, fill hourly_low where budget missing
df4['budget_clean'] = df4['budget'].fillna(
    df4['hourly_low'].fillna(0) * 40   # assume 40h for hourly jobs
)

# Extract skills from description using regex
def extract_skills(text):
    skill_pattern = re.compile(
        r'\b(python|react|node\.?js|javascript|typescript|sql|django|flask|'
        r'fastapi|aws|docker|git|html|css|java|php|ruby|vue|angular|'
        r'machine learning|data science|tensorflow|pytorch|pandas|numpy|'
        r'mongodb|postgresql|mysql|redis|graphql|rest api|flutter|swift)\b',
        re.IGNORECASE
    )
    return list(set(skill_pattern.findall(text.lower())))

df4['extracted_skills'] = df4['description'].apply(extract_skills)

# Classify difficulty based on budget
def classify_difficulty(budget):
    if budget == 0:
        return 'unknown'
    elif budget < 150:
        return 'starter'
    elif budget < 500:
        return 'easy'
    elif budget < 2000:
        return 'intermediate'
    else:
        return 'advanced'

df4['difficulty'] = df4['budget_clean'].apply(classify_difficulty)

# Keep only starter + easy for the launchpad
df4_launchpad = df4[df4['difficulty'].isin(['starter','easy'])].reset_index(drop=True)

print("df4 full shape      :", df4.shape)
print("df4 launchpad shape :", df4_launchpad.shape)
print("\nDifficulty distribution:")
print(df4['difficulty'].value_counts())
print("\nSample launchpad projects:")
print(df4_launchpad[['title','budget_clean','difficulty','extracted_skills']].head(5))

df4 full shape      : (53058, 12)
df4 launchpad shape : (28367, 12)

Difficulty distribution:
difficulty
starter         14590
intermediate    13988
easy            13777
unknown          8229
advanced         2474
Name: count, dtype: int64

Sample launchpad projects:
                                               title  budget_clean difficulty  \
0                                    SMMA Bubble App         400.0       easy   
1                   Want to fix the WordPress Plugin           5.0    starter   
2  need Portuguese writers who can understand and...         280.0       easy   
3  3D designer needed to create a 3D model of a c...          50.0    starter   
4                           US Located So-Me Manager         300.0       easy   

  extracted_skills  
0               []  
1               []  
2               []  
3               []  
4               []  


In [ ]:
# Cell 8 — Skill taxonomy from df3
from collections import defaultdict

category_skills = defaultdict(set)

for _, row in df3.iterrows():
    cat = row['job_category']
    for skill in row['skills_list']:
        if len(skill) > 2:
            category_skills[cat].add(skill)

# Convert to regular dict of sorted lists
skill_map = {cat: sorted(list(skills)) for cat, skills in category_skills.items()}

# Show top skills per category
for cat, skills in list(skill_map.items())[:3]:
    print(f"\n{cat}:")
    print(skills[:10])

print(f"\n✅ Skill taxonomy built: {len(skill_map)} categories")


Artificial Intelligence:
['###', '****', 'a.i.', 'a/b testing', 'aar and fra regulations', 'ability to efficiently operate a desktop computer', 'account management', 'account reconciliations', 'accounting', 'accounting tools and technology']

Big Data Engineer:
['****', '**technical skills**', '.net', '.net)', 'abap for bw', 'ability for continuous improvement', 'ability to work in a cross-functional team environment', 'ability to work with little direct supervision', 'active directory', 'activemq']

Business Analyst:
[') business analysis', ') deployment management', ') financial services', ') investment banking', ') qa coordination', ') sdlc', ') solution design', ') test case validation', ') uat coordination', '****']

✅ Skill taxonomy built: 25 categories


In [ ]:
# Cell 9 IMPROVED — Better beginner classifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# ── Re-engineer the label more precisely ──────────────────────────
# A true "launchpad beginner" has LOW jobs + LOW earnings + LOW rehire
df1['is_beginner_v2'] = (
    (df1['jobs_completed']   <= df1['jobs_completed'].quantile(0.35)) &
    (df1['earnings_usd']     <= df1['earnings_usd'].quantile(0.35))   &
    (df1['rehire_rate']      <= df1['rehire_rate'].quantile(0.40))
).astype(int)

print("Improved beginner label distribution:")
print(df1['is_beginner_v2'].value_counts())
print(f"Beginner ratio: {df1['is_beginner_v2'].mean():.1%}")

# ── Features ──────────────────────────────────────────────────────
features = [
    'jobs_completed', 'earnings_usd', 'hourly_rate',
    'job_success_rate', 'client_rating',
    'job_duration_days', 'rehire_rate'
]

# Add ratio features — these help a lot
df1['earn_per_job']    = df1['earnings_usd'] / (df1['jobs_completed'] + 1)
df1['success_x_rehire']= df1['job_success_rate'] * df1['rehire_rate'] / 100

features_v2 = features + ['earn_per_job', 'success_x_rehire']

X = df1[features_v2].fillna(0)
y = df1['is_beginner_v2']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Model: Gradient Boosting (better than RF for this) ────────────
pipeline_beginner = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GradientBoostingClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42
    ))
])

pipeline_beginner.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────
y_pred = pipeline_beginner.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

cv_scores = cross_val_score(pipeline_beginner, X, y, cv=5, scoring='f1')

print(f"\nTest Accuracy  : {acc*100:.1f}%")
print(f"Cross-val F1   : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=['Not Beginner', 'Beginner']
))

# ── Feature importance ────────────────────────────────────────────
importances = pd.Series(
    pipeline_beginner.named_steps['clf'].feature_importances_,
    index=features_v2
).sort_values(ascending=False)

print("Feature importances:")
print(importances.round(3))

# ── Quick sanity check ────────────────────────────────────────────
# A clear beginner (0 jobs, low earnings)
test_beg = pd.DataFrame([{
    'jobs_completed':0, 'earnings_usd':0, 'hourly_rate':8,
    'job_success_rate':0, 'client_rating':0,
    'job_duration_days':0, 'rehire_rate':0,
    'earn_per_job':0, 'success_x_rehire':0
}])
# An experienced freelancer
test_exp = pd.DataFrame([{
    'jobs_completed':200, 'earnings_usd':15000, 'hourly_rate':85,
    'job_success_rate':95, 'client_rating':4.9,
    'job_duration_days':30, 'rehire_rate':70,
    'earn_per_job':75, 'success_x_rehire':66.5
}])

print("\nSanity checks:")
print(f"  Clear beginner  → predicted: {'Beginner' if pipeline_beginner.predict(test_beg)[0] else 'Not Beginner'}")
print(f"  Experienced pro → predicted: {'Beginner' if pipeline_beginner.predict(test_exp)[0] else 'Not Beginner'}")

Improved beginner label distribution:
is_beginner_v2
0    1372
1      69
Name: count, dtype: int64
Beginner ratio: 4.8%

Test Accuracy  : 100.0%
Cross-val F1   : 0.985 ± 0.019

Classification Report:
              precision    recall  f1-score   support

Not Beginner       1.00      1.00      1.00       275
    Beginner       1.00      1.00      1.00        14

    accuracy                           1.00       289
   macro avg       1.00      1.00      1.00       289
weighted avg       1.00      1.00      1.00       289

Feature importances:
rehire_rate          0.565
jobs_completed       0.316
earnings_usd         0.118
client_rating        0.000
job_success_rate     0.000
job_duration_days    0.000
success_x_rehire     0.000
hourly_rate          0.000
earn_per_job         0.000
dtype: float64

Sanity checks:
  Clear beginner  → predicted: Beginner
  Experienced pro → predicted: Not Beginner


In [ ]:
# Cell 10 — Train project difficulty classifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# Use only rows where difficulty is known
df4_known = df4[df4['difficulty'] != 'unknown'].copy()

# Use title + first 300 chars of description as input text
df4_known['text_input'] = (
    df4_known['title'] + ' ' +
    df4_known['description'].str[:300]
)

# Binary: 1 = launchpad-suitable (starter/easy), 0 = too hard
df4_known['is_launchpad'] = df4_known['difficulty'].isin(
    ['starter','easy']
).astype(int)

X_text = df4_known['text_input']
y_diff  = df4_known['is_launchpad']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_text, y_diff, test_size=0.2, random_state=42, stratify=y_diff
)

pipeline_difficulty = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')),
    ('clf',   LogisticRegression(max_iter=500, class_weight='balanced'))
])

pipeline_difficulty.fit(X_tr, y_tr)

y_pred_d = pipeline_difficulty.predict(X_te)
print("Difficulty Classifier Accuracy:", round(accuracy_score(y_te, y_pred_d) * 100, 1), "%")
print("\nClassification Report:")
print(classification_report(y_te, y_pred_d, target_names=['Too Hard','Launchpad OK']))

Difficulty Classifier Accuracy: 74.2 %

Classification Report:
              precision    recall  f1-score   support

    Too Hard       0.63      0.74      0.68      3292
Launchpad OK       0.83      0.74      0.78      5674

    accuracy                           0.74      8966
   macro avg       0.73      0.74      0.73      8966
weighted avg       0.76      0.74      0.75      8966



In [ ]:
# Cell 11 — Match score function
def compute_match_score(freelancer_skills: list, project_skills: list) -> float:
    """
    Computes a 0-100 match score between freelancer skills
    and project required skills using Jaccard similarity.
    """
    if not project_skills:
        return 0.0

    f_set = set(s.lower().strip() for s in freelancer_skills)
    p_set = set(s.lower().strip() for s in project_skills)

    if not f_set:
        return 0.0

    intersection = f_set & p_set
    union        = f_set | p_set

    jaccard = len(intersection) / len(union)

    # Boost score if freelancer covers all required skills
    coverage = len(intersection) / len(p_set)
    score = (jaccard * 0.4 + coverage * 0.6) * 100

    return round(min(score, 100.0), 1)

# Quick test
freelancer = ['python', 'flask', 'sql', 'rest api']
project    = ['python', 'flask', 'rest api']
print(f"Test match score: {compute_match_score(freelancer, project)}%")

freelancer2 = ['react', 'css']
project2    = ['python', 'django', 'postgresql']
print(f"Low match score : {compute_match_score(freelancer2, project2)}%")

Test match score: 90.0%
Low match score : 0.0%


In [ ]:
# Cell 12 UPDATED — use pipeline_beginner
def recommend_launchpad_projects(
    freelancer_profile: dict,
    projects_df: pd.DataFrame,
    top_n: int = 5
) -> pd.DataFrame:

    # Build feature vector with new engineered features
    jobs      = freelancer_profile.get('jobs_completed', 0)
    earnings  = freelancer_profile.get('earnings_usd', 0)
    rate      = freelancer_profile.get('job_success_rate', 0)
    rehire    = freelancer_profile.get('rehire_rate', 0)

    features_vec = pd.DataFrame([{
        'jobs_completed'   : jobs,
        'earnings_usd'     : earnings,
        'hourly_rate'      : freelancer_profile.get('hourly_rate', 0),
        'job_success_rate' : rate,
        'client_rating'    : freelancer_profile.get('client_rating', 0),
        'job_duration_days': freelancer_profile.get('job_duration_days', 0),
        'rehire_rate'      : rehire,
        'earn_per_job'     : earnings / (jobs + 1),
        'success_x_rehire' : rate * rehire / 100,
    }])

    is_beginner = pipeline_beginner.predict(features_vec)[0]

    if not is_beginner:
        print("ℹ️  Freelancer is NOT a beginner — Launchpad not needed.")
        return pd.DataFrame()

    # Score projects
    results = []
    for _, proj in projects_df.iterrows():
        score = compute_match_score(
            freelancer_profile.get('skills', []),
            proj['extracted_skills']
        )
        results.append({
            'title'           : proj['title'],
            'difficulty'      : proj['difficulty'],
            'budget'          : round(proj['budget_clean'], 0),
            'match_score'     : score,
            'required_skills' : proj['extracted_skills'],
        })

    result_df = (
        pd.DataFrame(results)
        .sort_values('match_score', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    return result_df

# Test — beginner with python skills
test_beginner = {
    'skills'           : ['python', 'flask', 'rest api', 'sql'],
    'jobs_completed'   : 0,
    'earnings_usd'     : 0,
    'hourly_rate'      : 10,
    'job_success_rate' : 0,
    'client_rating'    : 0,
    'job_duration_days': 0,
    'rehire_rate'      : 0,
}

# Test — experienced (should be rejected from launchpad)
test_expert = {
    'skills'           : ['python', 'django', 'aws'],
    'jobs_completed'   : 150,
    'earnings_usd'     : 20000,
    'hourly_rate'      : 80,
    'job_success_rate' : 95,
    'client_rating'    : 4.8,
    'job_duration_days': 25,
    'rehire_rate'      : 65,
}

print("=== Beginner freelancer ===")
print(recommend_launchpad_projects(test_beginner, df4_launchpad, top_n=5).to_string(index=False))

print("\n=== Expert freelancer ===")
recommend_launchpad_projects(test_expert, df4_launchpad, top_n=5)

=== Beginner freelancer ===
                                              title difficulty  budget  match_score required_skills
           Developer for .Net, SQL, Python Projects       easy   400.0         80.0   [sql, python]
Python developer for resale marketplace webscraping       easy   360.0         80.0   [sql, python]
     Product analyst task(A/B testing, sql, python)    starter    40.0         80.0   [sql, python]
                            Geospatial Data Analyst       easy   400.0         80.0   [sql, python]
                       Selenium based flask project       easy   200.0         80.0 [flask, python]

=== Expert freelancer ===
ℹ️  Freelancer is NOT a beginner — Launchpad not needed.


""


In [ ]:
# Cell 13 UPDATED — Save models
os.makedirs('skillink_model', exist_ok=True)

joblib.dump(pipeline_beginner,       'skillink_model/clf_beginner.joblib')
joblib.dump(pipeline_difficulty,     'skillink_model/pipeline_difficulty.joblib')

import json
with open('skillink_model/skill_map.json', 'w') as f:
    json.dump(skill_map, f, indent=2)

print("✅ Saved:")
print("  skillink_model/clf_beginner.joblib")
print("  skillink_model/pipeline_difficulty.joblib")
print("  skillink_model/skill_map.json")

In [ ]:
# Cell 14 — Download to your computer
from google.colab import files

files.download('skillink_model/clf_beginner.joblib')
files.download('skillink_model/pipeline_difficulty.joblib')
files.download('skillink_model/skill_map.json')

print("✅ Download started — save these into Skillink-AI/skillink_model/")